# 다른 검증 영상에서도 0.5배 시간 연결이 도움이 될까?

지난 77프레임에서 정한 설정을 동결하고, 이전 연속 프레임 추적 비교에 포함되지 않은 b03·c04·c08의 준비~받기 171프레임을 평가합니다. 사람별 학습·검증 분할을 유지합니다. 같은 베이스라인 후보에서 **프레임별 최고 신뢰도 원판**과 **고신뢰 검출 뒤 최대 두 프레임까지의 저신뢰 연결**을 비교합니다. 선수별 실제 복구, 잘못 연결한 프레임, 최장 연속 미탐을 확인합니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

from wlcv.config import ANNOTATIONS, RUNS, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.temporal import evaluate_frames
from wlcv.evaluation.plate_track import KEY, BOX, select_centers, score_centers, summarize_centers

ranges = {"b03_sn_f_01": (8, 61), "c04_sn_f_01": (11, 78), "c08_sn_f_01": (10, 58)}
review_csv = ANNOTATIONS / "additional_temporal_review.csv"
review = pd.read_csv(review_csv)
expected = [(v, f) for v, (start, end) in ranges.items() for f in range(start, end + 1)]
assert list(review[KEY].itertuples(index=False, name=None)) == expected
assert review.box_source.eq("existing").sum() == 18
old_videos = pd.read_csv(ANNOTATIONS / "temporal_plate_centers.csv").video_id.unique()
assert set(ranges).isdisjoint(old_videos)
out = RUNS / "plate_baseline/report/additional_temporal_tracking"
out.mkdir(exist_ok=True, parents=True)
display(review.groupby(["video_id", "visibility"]).size().unstack(fill_value=0))

## 1. 모델을 적용하기 전 원판과 허브 확인

기존 박스 18개 사이의 원본 밝기 패턴을 앞뒤 프레임에서 맞추고 크기를 보간해 153개 초안을 만들었습니다. 전 프레임을 확대해 가까운 원판의 윤곽을 검수·보정했습니다. 허브는 돌출된 봉 끝을 피해 원판 면의 축 결합부를 확대 격자에서 한 번씩 판독했습니다. 흐린 프레임은 `approximate`이며 원본 픽셀 좌표로 기록했습니다. 이 평가용 자료는 YOLO 후보를 보기 전에 고정했습니다.

In [ ]:
def show_plate(ax, row, prediction=None):
    rgb = read_rgb(VIDEOS / f"{row.video_id}.mp4", row.frame_index)
    width, height = row.x2 - row.x1, row.y2 - row.y1
    pad = max(width, height) * .25
    left, right = max(0, int(row.x1-pad)), min(rgb.shape[1], int(row.x2+pad+1))
    top, bottom = max(0, int(row.y1-pad)), min(rgb.shape[0], int(row.y2+pad+1))
    ax.imshow(rgb[top:bottom, left:right], extent=(left-.5, right-.5, bottom-.5, top-.5))
    ax.add_patch(Rectangle((row.x1,row.y1), width,height, fill=False, ec="#ffd34b", lw=1.4))
    ax.plot(row.manual_x, row.manual_y, "+", c="#ffd34b", ms=11, mew=1.8)
    if prediction is not None and pd.notna(prediction.cx):
        ax.add_patch(Rectangle((prediction.x1,prediction.y1), prediction.x2-prediction.x1,
                               prediction.y2-prediction.y1, fill=False, ec="#34dce4", lw=1.3))
        ax.plot(prediction.cx, prediction.cy, "o", ms=7, mfc="none", mec="#34dce4")
    ax.set(title=f"{row.video_id[:3]} / frame {row.frame_index}", xticks=[], yticks=[])

fig, axes = plt.subplots(1, 3, figsize=(9, 3.4), layout="constrained")
for ax, (video, (start, end)) in zip(axes, ranges.items()):
    row = review[(review.video_id == video) & (review.frame_index == (start+end)//2)].iloc[0]
    show_plate(ax, row)
fig.suptitle("Reviewed plate and hub before inference")
plt.show()

## 2. 한 번 추론한 뒤 같은 후보로 두 방법 비교

Mac CPU, 입력 640, 후보 하한 0.001입니다. 두 방법 모두 신뢰도 0.10 이상이면 같은 최고 점수 후보를 선택합니다. 연결 방법은 고신뢰 후보가 없을 때만, 예상 위치에서 직전 박스 너비의 절반 이내에 있는 가장 가까운 후보를 찾습니다. 후보 선택에는 정답·수동 중심을 전달하지 않습니다. 선택한 박스의 정답 IoU ≥ 0.50을 정탐으로 판정합니다.

In [ ]:
weights = RUNS / "plate_baseline/weights/best.pt"
boxes, candidates = evaluate_frames(weights, review_csv, list(ranges))
pd.testing.assert_index_equal(boxes.set_index(KEY).index, review.set_index(KEY).index)
np.testing.assert_allclose(boxes[BOX], review[BOX], atol=1e-6, rtol=0)
checks = []
choices = select_centers(boxes[KEY], candidates[KEY + ["confidence"] + BOX],
                         distance_ratio=0.5, checks=checks)
detail = score_centers(choices, review[KEY + ["visibility", "manual_x", "manual_y"]], boxes)
summary = summarize_centers(detail)
for table, filename in [(candidates, "candidates"), (detail, "per_frame"), (summary, "summary")]:
    table.to_csv(out / f"{filename}.csv", index=False)
display(summary[["video_id", "method", "frames", "hits", "wrong_boxes", "longest_gap", "median_error_px"]].round(2))

## 3. 새 복구와 오연결이 실제로 발생한 횟수

`recovered`는 YOLO가 놓친 원판을 실제 후보로 다시 찾은 경우, `wrong_link`는 연결한 후보가 원판과 IoU 0.50을 충족하지 못한 경우입니다. `checks`는 시간 연결의 거리 조건을 실제 검사한 횟수입니다. 이 횟수가 0이면 이 구간에는 연결 규칙의 효과를 검증할 기회가 없었습니다.

In [ ]:
yolo = detail[detail.method == "YOLO"].set_index(KEY).sort_index()
past = detail[detail.method == "Past link"].set_index(KEY).sort_index()
paired = pd.DataFrame({"recovered": ~yolo.match & past.match,
                       "lost": yolo.match & ~past.match,
                       "linked": past.source.eq("linked"),
                       "wrong_link": past.source.eq("linked") & ~past.match})
totals = paired.groupby("video_id").sum()
totals["checks"] = [sum(row["video_id"] == video for row in checks) for video in totals.index]
totals["minimum_selected_confidence"] = yolo.confidence.groupby("video_id").min()
totals.to_csv(out / "paired_summary.csv")
display(totals.round(3))
display(past[paired.linked][["confidence", "iou", "error_px"]])

## 4. 신뢰도 기준과 허브 중심 오차 시각 검수

위 그래프는 영상별 최고 후보 신뢰도와 검출 기준, 아래 그래프는 두 방법의 수동 허브 중심 대비 오차입니다. 결측은 선을 끊습니다. 마지막 그림에서는 각 영상의 프레임별 YOLO 중심 오차가 가장 컸던 판독 가능 사례를 봅니다. 노란색은 검수 박스와 수동 중심, 청록색은 선택한 박스와 그 중심입니다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11.4, 5.7), sharex="col", sharey="row", layout="constrained")
for col, video in enumerate(ranges):
    base, linked = yolo.loc[video], past.loc[video]
    peak_scores = candidates[candidates.video_id == video].groupby("frame_index").confidence.max()
    axes[0,col].plot(base.index, peak_scores.reindex(base.index), color="#126e78")
    axes[0,col].axhline(.10, color="#a34c24", ls="--")
    axes[0,col].set(title=video[:3], ylim=(0,1.04))
    axes[1,col].plot(base.index, base.error_px, color="#293e53", lw=2, label="Frame YOLO")
    axes[1,col].plot(linked.index, linked.error_px, color="#e38232", lw=1.4, ls="--", label="Past link")
    axes[1,col].set(xlabel="Video frame", ylim=(0,None))
axes[0,0].set_ylabel("Top candidate confidence")
axes[1,0].set_ylabel("Hub center error (px)")
axes[1,0].legend(fontsize=8)
fig.suptitle("Confidence cutoff (0.10) and hub center error")
fig.savefig(out / "comparison.png", dpi=145, facecolor="white")
plt.show()

In [ ]:
lookup = review.set_index(KEY, drop=False)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.8), layout="constrained")
for ax, video in zip(axes, ranges):
    observed = yolo.loc[video].dropna(subset=["error_px"])
    if observed.empty:
        ax.set_axis_off()
        continue
    frame = observed.error_px.idxmax()
    prediction = observed.loc[frame]
    show_plate(ax, lookup.loc[(video, frame)], prediction)
    ax.set_xlabel(f"Conf {prediction.confidence:.3f}   IoU {prediction.iou:.3f}   Error {prediction.error_px:.1f} px")
fig.suptitle("Largest observed center error per video: manual (+) and YOLO (circle)")
fig.savefig(out / "reviewed_frames.png", dpi=145, facecolor="white")
plt.show()